# N064 · 子集、排列与组合回溯

**目标：** 区分位置选择、使用标记与起点约束。

**先修：** N062, N016。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 选择树；choose/explore/undo；路径拷贝；输出规模；完备性。

**配套讲解来源：** [同名逐章意见](../../comment/064_subsets_permutations_combinations.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的是三道"把所有方案都列出来"的枚举题，它们长得像但规则不同：

1. **子集**：给你一个数组（比如 `[1,2,3]`），你要列出它的所有子集。`[1,2,3]` 一共有 8 个子集：`[]`、`[1]`、`[2]`、`[3]`、`[1,2]`、`[1,3]`、`[2,3]`、`[1,2,3]`。为什么是 8 个？因为每个元素都有"要"或"不要"两种选择，3 个元素就是 2×2×2=8 种组合。
2. **排列**：给你 `[1,2,3]`，你要按顺序列出所有"排队方式"，一共 3!=6 个：`[1,2,3]`、`[1,3,2]`、`[2,1,3]`、`[2,3,1]`、`[3,1,2]`、`[3,2,1]`。每个元素必须出现且只出现一次。
3. **组合**：给你 n 和 k（比如 n=4、k=2），你要从 1..4 里选 2 个数，输出 6 种选法：`[1,2]`、`[1,3]`、`[1,4]`、`[2,3]`、`[2,4]`、`[3,4]`。注意 `[1,2]` 和 `[2,1]` 算同一个答案，只保留一种顺序。

这三道题都要求列出具体方案，而不是只计算方案数；因此必须访问每个要输出的方案。本章学的回溯（backtracking）就是"试一条路，走不通或走完了就退回来换一条"的通用套路。三道题的区别只在于"每一步的选择受什么限制"：子集是每个元素选或不选；排列是每层只能选还没用过的；组合是只能往后选，不许回头。

一个最小例子：输入 `nums=[1,2]`，`subsets` 输出 `[[], [2], [1], [1,2]]`（顺序后面手算解释），`permute` 输出 `[[1,2],[2,1]]`，`combine(2,1)` 输出 `[[1],[2]]`。

## 2. 基础知识：先读懂这些词

- **回溯（backtracking）**：一种"边构建边试探"的搜索方法。我们维护一条半成品答案（路径），每一步做一个选择并往下递归；递归返回后立刻撤销这个选择，恢复原状，再试下一个选择。你可以把它想象成走迷宫时沿路撒面包屑，走死胡同就顺着面包屑退回来。
- **选择树**：把"每一步的所有可能选择"画成一棵树，树根是空方案，每个叶子是一个完整方案。回溯就是对这棵树做深度优先遍历，本章 notebook 里的 `visit` 函数就是这棵树上的"走到某个节点"。
- **路径（path）**：当前这条半成品答案，是一个列表。往下走一层就 `append` 一个元素，退回来就 `pop`，这样同一个列表可以被所有分支反复复用。
- **choose / explore / undo**：回溯的三部曲，分别是"做选择"（append、置 used）、"递归探索"（visit）、"撤销选择"（pop、还原 used）。三步必须成对出现，少了 undo 状态就被污染了。
- **路径拷贝（`path[:]`）**：`path[:]` 会生成一份内容相同的**新列表**。收获答案时必须存拷贝，因为 `path` 这个列表本身后面还会被 pop 清空；如果直接存 `path` 的引用，最终所有"答案"都指向同一个被清空的列表。
- **used 标记**：排列题里用来记录"哪个元素已经在当前路径里"的布尔数组。它是为"每层只能选没用过的元素"这条规则服务的。
- **起点约束（start）**：组合题里"下一层只能从更大的下标开始选"的规则，它保证 `[1,2]` 只会以一种顺序被生成，天然避免重复。
- **输出规模**：子集有 2ⁿ 个、排列有 n! 个、组合有 C(n,k) 个。显式枚举算法至少要构造或输出全部答案，所以复杂度不可能低于输出规模，这一点决定了这类题的输入 n 通常很小。

## 3. 思路推导：从小例子开始

**Step 1：把"枚举"想象成"每个位置做一次选择"。** 我们站在第 i 个元素面前，问自己"这个元素要不要放进子集"，这就是一个二叉分支；所有分支展开成一棵树，叶子就是全部答案。

**Step 2：用递归函数走树，参数就是"现在考虑到哪个元素"。** 子集的 `visit(i)` 意思是"前 i 个元素的取舍已经定了，现在从第 i 个开始决定"。走到 `i==len(nums)` 说明所有元素都决定了，此时 `path` 就是一个完整子集。

**Step 3：排列加上"不能用过的"约束。** 每一层我们扫描所有元素，跳过 `used[i]` 为真的那些；选中的立刻标记，递归回来再取消标记。

**Step 4：组合加上"只能往后选"的约束。** `visit(start)` 里的 start 是"本层允许选的最小数值"，下一层传 `x+1`，于是生成的序列天然递增。

**Step 5：收获答案时存拷贝 `path[:]`。** 这是新手最容易踩的坑，第六节的测试专门验证了这一点。

**手算演示 1：`subsets([1,2])` 一轮一轮走。**

1. `visit(0)`：先走"不选 1"的分支（注意代码里 `visit(i+1)` 在前，所以"不选"先走）。
2. 进入 `visit(1)`：再走"不选 2"的分支 → `visit(2)`：i 等于 2 等于 len(nums)，收集 `path[:]`，此时 path 是空的，收集 `[]`。
3. 回到 `visit(1)`，走"选 2"分支：path 变成 `[2]` → `visit(2)` 收集 `[2]`，然后 pop 还原。
4. 回到 `visit(0)`，走"选 1"分支：path 变成 `[1]` → `visit(1)`：不选 2 → 收集 `[1]`；选 2 → path=`[1,2]` → 收集 `[1,2]`。
5. 最终输出顺序是 `[[], [2], [1], [1,2]]`。notebook “运行示例”部分 里把 `subsets([1,2,3])` 的 8 个叶子编了号列成表，你按同样方式推一遍，顺序一定对得上：先全不选的 `[]`，再只选末尾元素的，逐层往前。

**手算演示 2：`combine(4,2)`（对应 “运行示例”部分 第二张表"从1..4选2"）。**

1. 第一层：path 为空，`remaining=k-len(path)=2`，候选 x 从 `range(start, n-remaining+2)` 即 `range(1,4)` 取值，也就是 1、2、3。
2. 选 1：进入第二层，start=2，remaining=1，候选是 `range(2,5)` 即 2、3、4 → 得到 `[1,2]`、`[1,3]`、`[1,4]`。
3. 选 2：第二层候选 3、4 → `[2,3]`、`[2,4]`；选 3：候选 4 → `[3,4]`。
4. 一共 6 条，与表里的 6 行完全一致。

这里有个精妙的剪枝：为什么第一层不许选 4？如果你选了 4，后面一个数都没有了，但还差 1 个数没选，必然失败。公式 `n-remaining+2` 里，range 不含右端点，所以实际允许的最大值是 `n-remaining+1`，也就是"给后面还差的每个位置都至少留一个数"的临界点。

**手算演示 3：`permute([1,2])`。** 第一层 i=0：标记 used[0]，path=[1] → 第二层只有 i=1 可选 → path=[1,2]，长度等于 2，收集 `[1,2]`；全部撤销后第一层 i=1：同理收集 `[2,1]`。

## 4. 核心代码：subsets

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def subsets(nums):
    result = []
    path = []

    def visit(i):
        if i == len(nums):
            result.append(path[:])
            return
        visit(i + 1)
        path.append(nums[i])
        visit(i + 1)
        path.pop()
    visit(0)
    return result
```

### 逐段读懂代码

### 4.1 `subsets(nums)`：每个元素"选或不选"

```python
def subsets(nums):
    result = []
    path = []

    def visit(i):
        if i == len(nums):
            result.append(path[:])
            return
        visit(i + 1)
        path.append(nums[i])
        visit(i + 1)
        path.pop()
    visit(0)
    return result
```

- `result=[]; path=[]`：result 收集所有答案，path 是当前半成品；分号把两行并成一行，Python 里完全合法。
- `if i==len(nums): result.append(path[:]); return`：下标越界到"所有元素都考虑完"时，path 恰好是一个完整子集，此时存拷贝 `path[:]` 并停止递归。
- `visit(i+1)`：第一个分支——**不选** `nums[i]`，path 原封不动，直接看下一个元素。
- `path.append(nums[i]); visit(i+1); path.pop()`：第二个分支——**选** `nums[i]`：先放进路径，递归处理后面的元素，回来后 `path.pop()` 把它撤掉，让兄弟分支看到干净的 path。这就是 choose → explore → undo。

### 4.2 `permute(nums)`：每层选一个没用过的

```python
def permute(nums):
    result = []
    path = []
    used = [False] * len(nums)

    def visit():
        if len(path) == len(nums):
            result.append(path[:])
            return
        for i, x in enumerate(nums):
            if used[i]:
                continue
            used[i] = True
            path.append(x)
            visit()
            path.pop()
            used[i] = False
    visit()
    return result
```

- `used=[False]*len(nums)`：和 nums 等长的布尔数组，初始全部未使用。
- `if len(path)==len(nums)`：路径装满了 n 个元素就是一个完整排列；因为每层选一个、每层都跳过用过的，装满时必然每个元素恰好出现一次。
- `for i,x in enumerate(nums)` 与 `if used[i]: continue`：逐个检查候选，用过的直接跳过。
- `used[i]=True; path.append(x)` 是 choose，`visit()` 是 explore，`path.pop(); used[i]=False` 是 undo。注意撤销的顺序和做选择时对称：先 pop 路径再清标记，两组操作都要做全。

### 4.3 `combine(n,k)`：只能往后选，还带剪枝

```python
def combine(n, k):
    if n < 0 or k < 0:
        raise ValueError('nonnegative n,k required')
    result = []
    path = []

    def visit(start):
        if len(path) == k:
            result.append(path[:])
            return
        remaining = k - len(path)
        for x in range(start, n - remaining + 2):
            path.append(x)
            visit(x + 1)
            path.pop()
    if k <= n:
        visit(1)
    return result
```

- `if n<0 or k<0: raise ValueError(...)`：这是输入契约检查——n 和 k 都必须非负，传负数立刻报错，不让错误状态悄悄往下传。
- `if len(path)==k`：选够 k 个数就收集一条答案。
- `remaining=k-len(path)`：还差几个数没选。
- `for x in range(start,n-remaining+2)`：候选从 start 到 `n-remaining+1`（range 右端点开区间，所以写 +2）。上界就是第三节解释的剪枝：如果当前选的 x 太大，后面剩的数不够填满 remaining 个位置，这条分支必然死掉，不如一开始就别进去。
- `path.append(x); visit(x+1); path.pop()`：老三部曲；下一层的 start 是 `x+1`，保证选出的序列严格递增，`[1,2]` 永远不会以 `[2,1]` 的样子再出现一次。
- `if k<=n: visit(1)`：如果要选的比总共的还多（k>n），直接跳过，返回空列表，这又是一个"提前判断必然失败"的剪枝。

## 5. 分段实现：按顺序运行

**本章接口：** `subsets(nums)`、`permute(nums)`、`combine(n, k)`

### subsets

In [1]:
def subsets(nums):
    result = []
    path = []

    def visit(i):
        if i == len(nums):
            result.append(path[:])
            return
        visit(i + 1)
        path.append(nums[i])
        visit(i + 1)
        path.pop()
    visit(0)
    return result

### permute

In [2]:
def permute(nums):
    result = []
    path = []
    used = [False] * len(nums)

    def visit():
        if len(path) == len(nums):
            result.append(path[:])
            return
        for i, x in enumerate(nums):
            if used[i]:
                continue
            used[i] = True
            path.append(x)
            visit()
            path.pop()
            used[i] = False
    visit()
    return result

### combine

In [3]:
def combine(n, k):
    if n < 0 or k < 0:
        raise ValueError('nonnegative n,k required')
    result = []
    path = []

    def visit(start):
        if len(path) == k:
            result.append(path[:])
            return
        remaining = k - len(path)
        for x in range(start, n - remaining + 2):
            path.append(x)
            visit(x + 1)
            path.pop()
    if k <= n:
        visit(1)
    return result

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['叶节点序号','子集'],list(enumerate(subsets([1,2,3]))))
show_table(['组合序号','从1..4选2'],list(enumerate(combine(4,2))))
def trace_subsets(nums):
    """把选、不选、保存和撤销事件显式记录下来。"""
    path = []
    result = []
    events = []
    def visit(index):
        events.append((index, '进入', tuple(path)))
        if index == len(nums):
            result.append(path.copy())
            events.append((index, '保存副本', tuple(path)))
            return
        events.append((index, '不选当前元素', tuple(path)))
        visit(index + 1)
        path.append(nums[index])
        events.append((index, '选当前元素', tuple(path)))
        visit(index + 1)
        path.pop()
        events.append((index, '撤销选择', tuple(path)))
    visit(0)
    assert path == []
    return result, events

traced_subsets, backtrack_events = trace_subsets([1, 2])
assert traced_subsets == subsets([1, 2]) == [[], [2], [1], [1, 2]]
show_table(['待处理下标', '动作', 'path 快照'], backtrack_events)

叶节点序号,子集
0,[]
1,[3]
2,[2]
3,"[2, 3]"
4,[1]
5,"[1, 3]"
6,"[1, 2]"
7,"[1, 2, 3]"


组合序号,从1..4选2
0,"[1, 2]"
1,"[1, 3]"
2,"[1, 4]"
3,"[2, 3]"
4,"[2, 4]"
5,"[3, 4]"


待处理下标,动作,path 快照
0,进入,()
0,不选当前元素,()
1,进入,()
1,不选当前元素,()
2,进入,()
2,保存副本,()
1,选当前元素,"(2,)"
2,进入,"(2,)"
2,保存副本,"(2,)"
1,撤销选择,()


## 7. 正确性、适用条件与复杂度

每个答案对应唯一根到叶的选择序列。撤销步骤恢复父状态，使兄弟分支互不污染；组合严格递增索引给每个无序集合唯一编码。

**代价：** 子集O(n2ⁿ)、排列O(n·n!)、组合O(k·C(n,k))输出规模；工作栈与path为O(n)，不计输出。

### 展开理解

**完备性与唯一性。** 对元素互不相同的输入，每个子集对应唯一的一串“选或不选”的决策，递归访问全部决策序列，因此不漏不重。排列在每层选择一个尚未使用的位置；每个排列同样对应唯一决策路径。组合只从当前值往后选，使同一组数的不同顺序不再重复出现。含重复值的输入需要额外去重规则，本章不能自动保证值级别唯一。

**撤销与副本。** 递归返回时，`path.pop()` 和 `used` 的恢复把状态还原到进入分支前。相邻分支才不会相互污染。保存答案时要用 `path.copy()`：否则答案数组里多次存入的是同一个可变列表的引用。

**复杂度要算输出。** 子集时间为 O(n·2ⁿ)，排列为 O(n·n!)，组合为 O(k·C(n,k))；这里包含构造答案副本的成本。工作栈、当前路径与使用标记的额外空间是 O(n)，但完整返回值还需要与全部输出元素数量相当的空间。例如 10! 已经有 3,628,800 个排列，不能只看工作栈很小就断言内存没有问题。

**枚举与计数不同。** 只问方案数时，公式或动态规划可能避免逐个生成；要求列出所有方案时，算法必须付出构造输出的成本。输入规模变大后，应先估算输出数量，再决定是全部保留、逐个生成，还是改问计数问题。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条拆开看：

- `assert len(subsets([1,2,3]))==8`：正常值测试。3 个元素应该有 2³=8 个子集，这条只查数量，是对"每个元素独立取舍"的最直接验证。
- `assert len(permute([1,2,3]))==6`：正常值测试。3!=6，验证排列数量公式。
- `for n in range(7): for k in range(n+1): ...` 的双重循环：把 n=0..6、k=0..n 的所有组合小规模穷举一遍，检查三件事——`len(out)==math.comb(n,k)`（数量等于数学公式）、`len(set(map(tuple,out)))`（把每条答案变成元祖放进集合后数量不变，说明没有重复答案）、`all(len(x)==k and x==sorted(x) for x in out)`（每条答案长度恰好是 k 且严格递增）。这是用标准库当"标准答案"做交叉验证。
- `out=subsets([1,2]); assert len({id(x) for x in out})==len(out)`：**别名测试**，专门验证"存的是拷贝而不是引用"。`id(x)` 返回对象的身份号，如果 4 个答案都是同一个列表对象，集合里只会有 1 个 id，断言失败。这正对应练习 2 探讨的坑。
- `assert permute([])==[[]]`：边界值。空数组的"排列"是"一个空的排列"，所以答案是包含一个空列表的列表，而不是空列表本身。
- `assert combine(2,3)==[]`：特殊值。从 2 个数里选 3 个是不可能的，应返回空列表，验证了 `if k<=n` 这道闸门。

In [5]:
import math

assert len(subsets([1, 2, 3])) == 8

assert len(permute([1, 2, 3])) == 6

for n in range(7):
    for k in range(n + 1):
        out = combine(n, k)
        assert len(out) == math.comb(n, k) == len(set(map(tuple, out)))
        assert all((len(x) == k and x == sorted(x) for x in out))

out = subsets([1, 2])

assert len({id(x) for x in out}) == len(out)

assert permute([]) == [[]] and combine(2, 3) == []

# 用标准库独立枚举，比较方案本身，而不只比较数量。
import itertools
for n in range(6):
    items = list(range(n))
    expected = {combo for k in range(n + 1) for combo in itertools.combinations(items, k)}
    assert set(map(tuple, subsets(items))) == expected
    assert set(map(tuple, permute(items))) == set(itertools.permutations(items))
copy_check = subsets([1, 2])
unchanged = [part.copy() for part in copy_check[1:]]
copy_check[0].append(99)
assert copy_check[1:] == unchanged

print('N064: 所有本章断言通过')

N064: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 不使用库函数枚举。

**练习 2：** 解释保存path引用为何错误。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（不使用库函数枚举）**：思路是你自己写一个计数或枚举逻辑来代替 `math.comb` / `itertools`。提示：可以用"逐个元素选或不选"的递归数出子集数量，或者手写二进制计数法——0 到 2ⁿ-1 的每个整数的二进制位对应"第 i 个元素要不要"。写完先用 n=3、k=2 手算出答案（比如 C(3,2)=3），再用你的实现和本章接口对比。
- **练习 2（解释保存 path 引用为何错误）**：提示——`result.append(path)` 存进去的是"这个列表对象本身"而不是它的内容；递归结束后 path 会被一路 pop 成空列表，于是 result 里所有条目都指向这个空列表。手算示例：`subsets([1])` 正确答案 `[[],[1]]`，写错版本会得到 `[[],[]]`（两个都是同一个空列表）。验证方法就是第六节那条 id 测试。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def subsets(nums):
    result = []
    path = []

    def visit(i):
        if i == len(nums):
            result.append(path[:])
            return
        visit(i + 1)
        path.append(nums[i])
        visit(i + 1)
        path.pop()
    visit(0)
    return result

def permute(nums):
    result = []
    path = []
    used = [False] * len(nums)

    def visit():
        if len(path) == len(nums):
            result.append(path[:])
            return
        for i, x in enumerate(nums):
            if used[i]:
                continue
            used[i] = True
            path.append(x)
            visit()
            path.pop()
            used[i] = False
    visit()
    return result

def combine(n, k):
    if n < 0 or k < 0:
        raise ValueError('nonnegative n,k required')
    result = []
    path = []

    def visit(start):
        if len(path) == k:
            result.append(path[:])
            return
        remaining = k - len(path)
        for x in range(start, n - remaining + 2):
            path.append(x)
            visit(x + 1)
            path.pop()
    if k <= n:
        visit(1)
    return result

# 示例与回归测试
import math

assert len(subsets([1, 2, 3])) == 8

assert len(permute([1, 2, 3])) == 6

for n in range(7):
    for k in range(n + 1):
        out = combine(n, k)
        assert len(out) == math.comb(n, k) == len(set(map(tuple, out)))
        assert all((len(x) == k and x == sorted(x) for x in out))

out = subsets([1, 2])

assert len({id(x) for x in out}) == len(out)

assert permute([]) == [[]] and combine(2, 3) == []

# 用标准库独立枚举，比较方案本身，而不只比较数量。
import itertools
for n in range(6):
    items = list(range(n))
    expected = {combo for k in range(n + 1) for combo in itertools.combinations(items, k)}
    assert set(map(tuple, subsets(items))) == expected
    assert set(map(tuple, permute(items))) == set(itertools.permutations(items))
copy_check = subsets([1, 2])
unchanged = [part.copy() for part in copy_check[1:]]
copy_check[0].append(99)
assert copy_check[1:] == unchanged

print('N064: 所有本章断言通过')

N064: 所有本章断言通过


## 11. 代表题与迁移

- **78. Subsets**：这题就是本章 `subsets` 的原题，练的是"每个位置选或不选"的二叉选择树和 path 拷贝。
- **46. Permutations**：对应 `permute`，练的是 used 标记挡住"同一层重复选同一个元素"。
- **77. Combinations**：对应 `combine`，练的是 start 起点约束和"给剩余位置留够数字"的剪枝。

三道题共用同一副"choose → explore → undo"骨架，你能把本章三个函数互相改写出来，这组题就算吃透了。

**题面入口：** [78](https://leetcode.com/problems/subsets/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。